<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Import Libraries</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Import Libraries
    </h1>
</div>


In [ ]:
import sys
import os
sys.path.append(os.path.abspath("../../.."))

from config.spark_config import SparkConfig
from utils.logger import LoggerFactory
from config.io_config import *
from app.platform_app import PlatformApp
from utils.data_quality import *
from utils.data_cleaning import *
from utils.utils import *
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Set up</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Set up
    </h1>
</div>


In [13]:
# Initialize shared logger (all logs in this run go to the same file: etl_<run_id>.log)
logger = LoggerFactory.setup_logger(name="ETL", log_dir=LOG_DIR)

# Create Spark session with logging enabled (for tracing Spark-related operations)
spark = SparkConfig.create_spark(app_name="Paypal Analytic", logger=logger, use_databricks=True)

# Initialize main application with Spark and logger (used across ETL pipeline)
app = PlatformApp(spark=spark, logger=logger, catalog_name="paypal_analytic")

2026-04-07 23:03:45 | INFO     | ETL | logger.py:113 | Logger initialized | level=DEBUG | file=C:/01_Data/05-data-engineer-bootcamp/03_paypal_databricks/logs\etl_20260401_193104_713773.log
2026-04-07 23:04:07 | INFO     | ETL | spark_config.py:89 | Connected to Databricks via Spark Connect.
2026-04-07 23:04:07 | INFO     | ETL | platform_app.py:44 | Initializing Data Platform...
2026-04-07 23:04:07 | INFO     | ETL | platform_app.py:50 | Spark session initialized


<!-- Include Google Fonts for a modern font -->
<link href="https://fonts.googleapis.com/css2?family=Roboto:wght@700&display=swap" rel="stylesheet">

# <span style="color:transparent;">Silver</span>

<div style="
    border-radius: 15px; 
    border: 2px solid #003366; 
    padding: 10px; 
    background: linear-gradient(135deg, #3a0ca3, #7209b7 30%, #f72585 80%);
    text-align: center; 
    box-shadow: 0px 4px 8px rgba(0, 0, 0, 0.5);
">
    <h1 style="
        color: #fff;
        text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7);
        font-weight: bold;
        margin-bottom: 10px;
        font-size: 36px;
        font-family: 'Roboto', sans-serif;
        letter-spacing: 1px;
    ">
        Silver
    </h1>
</div>


In [14]:
df_bronze_incentive = spark.sql(f"SELECT * FROM {BRONZE_TRANSACTIONS}")
df_bronze_incentive.show(n=10, truncate=False)

+-----------------+----------------------+------------------------------+------------------------------+----------------------------------------------+------------------+-------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------+----------------------------------------------+----------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------

## Transformations

### Trim spaces

In [15]:
# Remove those trim values
df_bronze_incentive = clean_dataframe(df=df_bronze_incentive)
df_bronze_incentive.show(n=10, truncate=False)

+-----------------+----------------------+------------------------------+------------------------------+----------------------------------------------+------------------+-------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------+----------------------------------------------+----------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------

### Select Features

In [16]:
df_silver_incentive = df_bronze_incentive.select("transaction_id", "transaction_event_code",
                                                 "transaction_updated_date", "incentive_info", "elton_created_at", "dt", "hour")
df_silver_incentive.show(n=10, truncate=False)

+-----------------+----------------------+------------------------------+--------------+------------------------------+----------+----+
|transaction_id   |transaction_event_code|transaction_updated_date      |incentive_info|elton_created_at              |dt        |hour|
+-----------------+----------------------+------------------------------+--------------+------------------------------+----------+----+
|8F987681LS7188052|T0006                 |2023-10-01 01:02:44.000000 UTC|{}            |2024-03-07 02:36:03.253095 UTC|2024-03-07|2   |
|95328985WP2643805|T0006                 |2023-10-01 01:17:17.000000 UTC|{}            |2024-03-07 02:36:03.253095 UTC|2024-03-07|2   |
|9P744271DM206464L|T1106                 |2023-10-01 01:49:31.000000 UTC|{}            |2024-03-07 02:36:03.253095 UTC|2024-03-07|2   |
|8V050773ME0194317|T1111                 |2023-10-01 01:49:31.000000 UTC|{}            |2024-03-07 02:36:03.253095 UTC|2024-03-07|2   |
|1MY863496S997480P|T0006                 |2023-1

### Duplicates

In [17]:
df_silver_incentive = dedup(
    df_silver_incentive,
    dedup_cols=["transaction_id", "transaction_event_code"],
    order_cols=["transaction_updated_date", "dt", "hour", "elton_created_at"],
    logger=logger
)

2026-04-07 23:04:44 | INFO     | ETL | utils.py:156 | Starting deduplication
2026-04-07 23:04:44 | INFO     | ETL | utils.py:157 | Dedup columns: ['transaction_id', 'transaction_event_code']
2026-04-07 23:04:44 | INFO     | ETL | utils.py:158 | Order columns: ['transaction_updated_date', 'dt', 'hour', 'elton_created_at']
2026-04-07 23:04:44 | INFO     | ETL | utils.py:176 | Order direction (desc): [True, True, True, True]
2026-04-07 23:04:44 | INFO     | ETL | utils.py:177 | Nulls last: True
2026-04-07 23:04:46 | INFO     | ETL | utils.py:184 | Input row count: 4792
2026-04-07 23:04:49 | INFO     | ETL | utils.py:219 | Output row count after dedup: 4002
2026-04-07 23:04:49 | INFO     | ETL | utils.py:220 | Removed duplicate rows: 790
2026-04-07 23:04:49 | INFO     | ETL | utils.py:221 | Deduplication completed


### Extract Data

In [18]:
# Define schema for each item in cart
incentive_schema = StructType([
    StructField("incentive_details", ArrayType(StructType([
        StructField("incentive_code", StringType()),
        StructField("incentive_program_code", StringType()),
        StructField("incentive_amount", StructType([
            StructField("currency_code", StringType()),
            StructField("value", StringType())
        ]))
    ])))
])

# Parse JSON -> struct (avoid multiple parsing)
df_parsed = df_silver_incentive.withColumn(
    "incentive",
    F.from_json(F.col("incentive_info"), incentive_schema)
)

# Explode array -> each row = 1 item
df_exploded = df_parsed.withColumn(
    "inc",
    F.explode_outer(F.col("incentive.incentive_details"))
)

# Preview result
df_exploded.show(n=10, truncate=False)

+-----------------+----------------------+------------------------------+--------------+------------------------------+----------+----+---------+----+
|transaction_id   |transaction_event_code|transaction_updated_date      |incentive_info|elton_created_at              |dt        |hour|incentive|inc |
+-----------------+----------------------+------------------------------+--------------+------------------------------+----------+----+---------+----+
|00010537PY789931F|T2103                 |2024-01-04 21:55:29.000000 UTC|{}            |2024-03-07 02:36:03.253095 UTC|2024-03-07|2   |{NULL}   |NULL|
|00A016522M383225C|T0006                 |2024-01-15 21:09:07.000000 UTC|{}            |2024-03-07 02:36:03.253095 UTC|2024-03-07|2   |{NULL}   |NULL|
|00A4466720602011T|T0200                 |2024-02-05 14:22:17.000000 UTC|{}            |2024-03-07 02:36:03.253095 UTC|2024-03-07|2   |{NULL}   |NULL|
|00D48250551281811|T0200                 |2024-02-14 12:34:28.000000 UTC|{}            |2024-0

In [19]:
df_silver_incentive_final = df_exploded.select(
    "transaction_id",
    "transaction_event_code",

    # Standardize timestamp
    parse_timestamp(F.col("transaction_updated_date")).alias("transaction_updated_date"),

    # incentive_code: trim -> blank -> NULL -> "Unknown"
    F.coalesce(
        F.when(F.trim(F.col("inc.incentive_code")) == "", None)
         .otherwise(F.trim(F.col("inc.incentive_code"))),
        F.lit("Unknown")
    ).alias("incentive_code"),

    # incentive_program_code: trim -> blank -> NULL -> "Unknown"
    F.coalesce(
        F.when(F.trim(F.col("inc.incentive_program_code")) == "", None)
         .otherwise(F.trim(F.col("inc.incentive_program_code"))),
        F.lit("Unknown")
    ).alias("incentive_program_code"),

    # currency_code: trim -> blank -> NULL -> "USD"
    F.coalesce(
        F.when(F.trim(F.col("inc.incentive_amount.currency_code")) == "", None)
         .otherwise(F.trim(F.col("inc.incentive_amount.currency_code"))),
        F.lit("USD")
    ).alias("currency_code"),

    # incentive_amount_value: cast to decimal
    F.col("inc.incentive_amount.value").cast("decimal(18,2)").alias("incentive_amount_value"),

    # Metadata timestamps
    parse_timestamp(F.col("elton_created_at")).alias("elton_created_at"),
    F.col("dt").cast("date").alias("dt"),
    F.col("hour").cast("int").alias("hour")
) \
.filter(F.col("transaction_id").isNotNull()) \
.withColumn("process_timestamp", F.date_trunc("second", F.current_timestamp()))

# Preview result
df_silver_incentive_final.show(n=10, truncate=False)

+-----------------+----------------------+------------------------+--------------+----------------------+-------------+----------------------+-------------------+----------+----+-------------------+
|transaction_id   |transaction_event_code|transaction_updated_date|incentive_code|incentive_program_code|currency_code|incentive_amount_value|elton_created_at   |dt        |hour|process_timestamp  |
+-----------------+----------------------+------------------------+--------------+----------------------+-------------+----------------------+-------------------+----------+----+-------------------+
|00010537PY789931F|T2103                 |2024-01-04 21:55:29     |Unknown       |Unknown               |USD          |NULL                  |2024-03-07 02:36:03|2024-03-07|2   |2026-04-07 16:04:52|
|00A016522M383225C|T0006                 |2024-01-15 21:09:07     |Unknown       |Unknown               |USD          |NULL                  |2024-03-07 02:36:03|2024-03-07|2   |2026-04-07 16:04:52|
|00A4

### Transformed data to Silver Layer

In [20]:
if not spark.catalog.tableExists(SILVER_PATH_DISPUTED_PP01_INCENTIVE):
    logger.info("Silver disputed pp01 incentive table not found. Creating new table...")
    df_silver_incentive_final.write.format("delta") \
                   .option("delta.enableChangeDataFeed", "true") \
                   .option("mergeSchema", "true") \
                   .mode("append") \
                   .saveAsTable(SILVER_PATH_DISPUTED_PP01_INCENTIVE)
    logger.info("Silver disputed pp01 incentive table created successfully")
else:
    logger.info("Silver disputed pp01 incentive table exists. Performing upsert...")
    upsert(spark=spark, df=df_silver_incentive_final, key_cols=["transaction_id", "transaction_event_code"],
           table=SILVER_TABLE_DISPUTED_PP01_INCENTIVE, cdc="transaction_updated_date",
           name_catalog=app.catalog_name, name_schema=SCHEMA_SILVER, logger=logger)
    logger.info("Upsert completed successfully")

2026-04-07 23:04:54 | INFO     | ETL | 733719856.py:10 | Silver disputed pp01 incentive table exists. Performing upsert...
2026-04-07 23:04:54 | INFO     | ETL | utils.py:315 | Starting UPSERT into paypal_analytic.silver.disputed_pp01_incentive
2026-04-07 23:05:14 | INFO     | ETL | utils.py:345 | UPSERT completed successfully: paypal_analytic.silver.disputed_pp01_incentive
2026-04-07 23:05:14 | INFO     | ETL | 733719856.py:14 | Upsert completed successfully


In [21]:
app.stop()

2026-04-07 23:05:14 | INFO     | ETL | platform_app.py:259 | Stopping Spark session...
2026-04-07 23:05:14 | INFO     | ETL | platform_app.py:261 | Spark stopped.
